In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
import warnings
import os

warnings.filterwarnings('ignore')

# 1. Load Data
def load_data():
    print("Loading data...")
    base_path = '/kaggle/input/store-sales-time-series-forecasting/'
    if not os.path.exists(base_path + 'train.csv'):
        base_path = '/kaggle/input/competitions/store-sales-time-series-forecasting/'
        
    train = pd.read_csv(base_path + 'train.csv', parse_dates=['date'])
    test = pd.read_csv(base_path + 'test.csv', parse_dates=['date'])
    stores = pd.read_csv(base_path + 'stores.csv')
    oil = pd.read_csv(base_path + 'oil.csv', parse_dates=['date'])
    holidays = pd.read_csv(base_path + 'holidays_events.csv', parse_dates=['date'])
    transactions = pd.read_csv(base_path + 'transactions.csv', parse_dates=['date'])
    return train, test, stores, oil, holidays, transactions

# 2. Granular Feature Engineering
def prepare_features(train_skeleton, test_skeleton, stores, oil, holidays, transactions=None):
    print("Preparing features...")
    
    stores = stores.rename(columns={'type': 'store_type'})
    holidays = holidays.rename(columns={'type': 'holiday_type'})
    
    # Oil processing
    oil['date'] = pd.to_datetime(oil['date'])
    oil = oil.set_index('date').resample('D').mean().interpolate(method='linear').reset_index()
    for w in [1, 7, 14]:
        oil[f'oil_lags_{w}'] = oil['dcoilwtico'].shift(w)
        oil[f'oil_rolling_{w}'] = oil['dcoilwtico'].rolling(w).mean()
    
    # Holiday Deduplication
    holidays = holidays[holidays['transferred'] == False]
    nat_hol = holidays[holidays['locale'] == 'National'].drop_duplicates('date')
    reg_hol = holidays[holidays['locale'] == 'Regional'].drop_duplicates(['date', 'locale_name'])
    loc_hol = holidays[holidays['locale'] == 'Local'].drop_duplicates(['date', 'locale_name'])
    
    # Combine train and test skeleton
    df = pd.concat([train_skeleton, test_skeleton], axis=0)
    df['date'] = pd.to_datetime(df['date'])
    
    # Merge datasets
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    
    # Merge holidays
    df = df.merge(nat_hol[['date', 'holiday_type']], on='date', how='left').rename(columns={'holiday_type': 'nat_hol_type'})
    reg_hol = reg_hol.rename(columns={'locale_name': 'state'})
    df = df.merge(reg_hol[['date', 'state', 'holiday_type']], on=['date', 'state'], how='left').rename(columns={'holiday_type': 'reg_hol_type'})
    loc_hol = loc_hol.rename(columns={'locale_name': 'city'})
    df = df.merge(loc_hol[['date', 'city', 'holiday_type']], on=['date', 'city'], how='left').rename(columns={'holiday_type': 'loc_hol_type'})
    
    # Transactions merge (if provided)
    if transactions is not None:
        trans_agg = transactions.groupby(['date', 'store_nbr'])['transactions'].sum().reset_index()
        df = df.merge(trans_agg, on=['date', 'store_nbr'], how='left')
        df['transactions'] = df['transactions'].fillna(0)
    
    # Time features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['wages_day'] = ((df['day_of_month'] == 15) | (df['date'].dt.is_month_end)).astype(int)
    
    # Categorical encoding
    le = LabelEncoder()
    cat_cols = ['city', 'state', 'store_type', 'cluster', 'nat_hol_type', 'reg_hol_type', 'loc_hol_type']
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    
    if 'family' in df.columns:
        df['family_encoded'] = le.fit_transform(df['family'].astype(str))
        
    return df

# 3. 2-Stage Model Execution
def main():
    train, test, stores, oil, holidays, transactions = load_data()
    
    # --- STAGE 1: Predict Transactions ---
    print("Stage 1: Forecasting Transactions...")
    train_dates = train[['date', 'store_nbr']].drop_duplicates()
    test_dates = test[['date', 'store_nbr']].drop_duplicates()
    
    df_trans = prepare_features(train_dates, test_dates, stores, oil, holidays)
    trans_clean = transactions.groupby(['date', 'store_nbr'])['transactions'].sum().reset_index()
    df_trans = df_trans.merge(trans_clean, on=['date', 'store_nbr'], how='left')
    df_trans['transactions'] = np.log1p(df_trans['transactions'].fillna(0))
    
    df_trans_train = df_trans[df_trans['date'] < '2017-08-16'].sort_values(['date', 'store_nbr'])
    y_trans = df_trans_train.set_index(['date', 'store_nbr'])['transactions'].unstack('store_nbr').fillna(0)
    y_trans.index = pd.to_datetime(y_trans.index)
    y_trans = y_trans.asfreq('D').fillna(0)
    
    fourier = CalendarFourier(freq="W", order=4)
    dp = DeterministicProcess(index=y_trans.index, constant=True, order=1, additional_terms=[fourier], drop=True)
    X_trans_1 = dp.in_sample()
    X_trans_test_1 = dp.out_of_sample(steps=16)
    
    model_trans_1 = Ridge(alpha=0.5)
    model_trans_1.fit(X_trans_1, y_trans)
    
    # Re-align Stage 1 Features
    df_trans_train_aligned = df_trans_train.set_index(['date', 'store_nbr']).reindex(
        pd.MultiIndex.from_product([y_trans.index, y_trans.columns], names=['date', 'store_nbr'])
    ).reset_index()
    
    features_trans = [col for col in df_trans.columns if col not in ['date', 'transactions', 'dcoilwtico']]
    X_trans_2 = df_trans_train_aligned[features_trans].fillna(0)
    y_trans_resid = y_trans.sort_index() - pd.DataFrame(model_trans_1.predict(X_trans_1), index=y_trans.index, columns=y_trans.columns)
    y_trans_resid = y_trans_resid.stack('store_nbr')
    
    model_trans_2 = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.1, random_state=42, verbosity=-1)
    model_trans_2.fit(X_trans_2, y_trans_resid)
    
    X_trans_test_2 = df_trans[df_trans['date'] >= '2017-08-16'].sort_values(['date', 'store_nbr'])[features_trans].fillna(0)
    y_trans_pred = pd.DataFrame(model_trans_1.predict(X_trans_test_1), index=X_trans_test_1.index, columns=y_trans.columns).stack('store_nbr')
    y_trans_pred += model_trans_2.predict(X_trans_test_2)
    y_trans_pred = np.expm1(y_trans_pred).reset_index()
    y_trans_pred.columns = ['date', 'store_nbr', 'transactions']
    
    # --- STAGE 2: Predict Sales ---
    print("Stage 2: Forecasting Sales...")
    full_transactions = pd.concat([trans_clean, y_trans_pred], axis=0).drop_duplicates(['date', 'store_nbr'])
    train_2017 = train[train['date'] >= '2017-01-01']
    df_sales = prepare_features(train_2017, test, stores, oil, holidays, full_transactions)
    df_sales['sales'] = np.log1p(df_sales['sales'].fillna(0))
    
    # CRITICAL: Create Lags BEFORE slicing the training set
    print("Creating Sales Lags...")
    for lag in [16, 17, 18, 28]:
        df_sales[f'lag_{lag}'] = df_sales.groupby(['store_nbr', 'family'])['sales'].shift(lag)
    
    # Slice Training Data AFTER lag creation
    df_sales_train = df_sales[df_sales['date'] < '2017-08-16'].sort_values(['date', 'store_nbr', 'family'])
    y_sales = df_sales_train.set_index(['date', 'store_nbr', 'family'])['sales'].unstack(['store_nbr', 'family']).fillna(0)
    y_sales.index = pd.to_datetime(y_sales.index)
    y_sales = y_sales.asfreq('D').fillna(0)
    
    dp_sales = DeterministicProcess(index=y_sales.index, constant=True, order=1, additional_terms=[fourier], drop=True)
    X_sales_1 = dp_sales.in_sample()
    X_sales_test_1 = dp_sales.out_of_sample(steps=16)
    
    model_sales_1 = Ridge(alpha=0.75)
    model_sales_1.fit(X_sales_1, y_sales)
    
    # Re-align Stage 2 Features
    excluded = ['id', 'date', 'sales', 'dcoilwtico', 'family']
    features_sales = [col for col in df_sales.columns if col not in excluded]
    
    df_sales_train_aligned = df_sales_train.set_index(['date', 'store_nbr', 'family']).reindex(
        pd.MultiIndex.from_product([y_sales.index, y_sales.columns.get_level_values(0).unique(), y_sales.columns.get_level_values(1).unique()], names=['date', 'store_nbr', 'family'])
    ).reset_index()
    
    X_sales_2 = df_sales_train_aligned[features_sales].fillna(0)
    y_sales_resid = y_sales.sort_index() - pd.DataFrame(model_sales_1.predict(X_sales_1), index=y_sales.index, columns=y_sales.columns)
    y_sales_resid = y_sales_resid.stack(['store_nbr', 'family'])
    
    print(f"Aligning Stage 2: X shape {X_sales_2.shape}, y shape {y_sales_resid.shape}")
    model_sales_2 = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.02, num_leaves=127, random_state=42, verbosity=-1, device="gpu")
    model_sales_2.fit(X_sales_2, y_sales_resid)
    
    X_sales_test_2 = df_sales[df_sales['date'] >= '2017-08-16'].sort_values(['date', 'store_nbr', 'family'])[features_sales].fillna(0)
    y_sales_pred = pd.DataFrame(model_sales_1.predict(X_sales_test_1), index=X_sales_test_1.index, columns=y_sales.columns).stack(['store_nbr', 'family'])
    y_sales_pred += model_sales_2.predict(X_sales_test_2)
    
    y_submit = y_sales_pred.reset_index()
    y_submit.columns = ['date', 'store_nbr', 'family', 'sales']
    submission = test.merge(y_submit, on=['date', 'store_nbr', 'family'], how='left')
    submission['sales'] = np.expm1(submission['sales']).clip(0, None)
    submission.loc[submission['date'].dt.dayofyear == 1, 'sales'] = 0
    submission[['id', 'sales']].to_csv('submission.csv', index=False)
    print("Success! Final submission saved.")

if __name__ == "__main__":
    main()


Loading data...
Stage 1: Forecasting Transactions...
Preparing features...
Stage 2: Forecasting Sales...
Preparing features...
Creating Sales Lags...
Aligning Stage 2: X shape (404514, 27), y shape (404514,)


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Success! Final submission saved.
